In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
len(v1)

384

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
len(dv)

384

In [6]:
v1.dot(dv)

0.32332402

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

0.019730492

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
from tqdm.auto import tqdm

In [12]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
X.shape

(1350, 384)

In [16]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [21]:
# scores = [v_query.dot(X[i]) for i in range(len(X))] -> same code but dot is faster
scores = X.dot(v_query)

In [23]:
idx = np.argmax(scores)
idx, scores[idx]

(2, 0.7629411)

In [24]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [29]:
# top5 = np.argsort(scores)[-5:]
# top5 = top5[::-1]
# top5
top5 = np.argsort(-scores)[:5]

In [26]:
# scores[top5]

array([  2, 625, 907, 538,   7])